# Phase 13: Anomaly Detection via Deep Autoencoder & Isolation Forest
### Smart Grid Cyber-Physical Intrusion Detection

In real-world smart grids, attackers constantly invent novel techniques that have no prior training signatures.
**Semi-supervised anomaly detection** models normal operational telemetry (`Normal` baseline only) and flags any statistical or physical deviations.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import load_data4cyber, get_preprocessor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on Device: {DEVICE}")


## 1. Load Data & Isolate Pure Normal Baseline
Scenario S0 represents uncontaminated benign DER operations.


In [ ]:
data = load_data4cyber(random_state=42)
prep = get_preprocessor(scaling=True)

X_tr_sc = prep.fit_transform(data['X_train'])
X_te_sc = prep.transform(data['X_test'])

benign_idx = list(data['label_encoder'].classes_).index('benign')

# Training set: ONLY benign normal data
X_norm_train = X_tr_sc[data['y_train_family'] == benign_idx]
print(f"Pure Normal Training Samples: {len(X_norm_train)}")

# Test set: Both normal and attack data
y_test_binary = data['y_test_binary']
print(f"Test Set: {len(X_te_sc)} (Normal: {np.sum(y_test_binary == 0)}, Attack: {np.sum(y_test_binary == 1)})")


## 2. Define & Train Deep Autoencoder


In [ ]:
class GridAutoencoder(nn.Module):
    def __init__(self, in_features, latent_dim=16):
        super(GridAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, latent_dim),
            nn.LeakyReLU(0.1)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, in_features)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = GridAutoencoder(X_norm_train.shape[1], latent_dim=16).to(DEVICE)
crit = nn.MSELoss()
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)

tx_norm = torch.tensor(X_norm_train, dtype=torch.float32)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(tx_norm), batch_size=64, shuffle=True)

print("Training Autoencoder on Normal Baseline...")
ae.train()
for ep in range(20):
    for (bx,) in loader:
        bx = bx.to(DEVICE)
        opt.zero_grad()
        loss = crit(ae(bx), bx)
        loss.backward()
        opt.step()

# Compute reconstruction errors on Normal Training to set threshold (95th percentile)
ae.eval()
with torch.no_grad():
    recon_tr = ae(tx_norm.to(DEVICE))
    err_tr = torch.mean((tx_norm.to(DEVICE) - recon_tr)**2, dim=1).cpu().numpy()

threshold = np.percentile(err_tr, 95.0)
print(f"Calibrated Anomaly Threshold (95th percentile of normal): {threshold:.5f}")


## 3. Evaluate Anomaly Detection on Test Set


In [ ]:
with torch.no_grad():
    tx_te = torch.tensor(X_te_sc, dtype=torch.float32).to(DEVICE)
    recon_te = ae(tx_te)
    err_te = torch.mean((tx_te - recon_te)**2, dim=1).cpu().numpy()

preds = (err_te >= threshold).astype(int)

acc = accuracy_score(y_test_binary, preds)
rec = recall_score(y_test_binary, preds)
f1 = f1_score(y_test_binary, preds, average='macro')
fpr = np.mean(preds[y_test_binary == 0])

print(f"Anomaly Detection Performance:")
print(f"  Accuracy: {acc:.4f} | Macro F1: {f1:.4f} | Attack Recall: {rec:.4f} | False Alarm Rate (FPR): {fpr:.4f}")


## 4. Plot Reconstruction Error Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(err_te[y_test_binary == 0], label='Normal Operational Baseline', fill=True, color='green', common_norm=False)
sns.kdeplot(err_te[y_test_binary == 1], label='Cyberattack Deviations', fill=True, color='red', common_norm=False)
plt.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.4f})')
plt.title("Autoencoder Reconstruction Residual Distribution", fontweight='bold')
plt.xlabel("Mean Squared Reconstruction Error")
plt.ylabel("Density")
plt.xlim(0, np.percentile(err_te, 99))
plt.legend()
plt.tight_layout()
plt.show()
